<a href="https://colab.research.google.com/github/sean838432/TdAI/blob/main/data_download/NBM_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""

This script loads in NBM station data over a specified time range
from the the NOAA NBM GRIB2 PDS S3 AWS bucket and saves the data to individual
files based on the NBM run (e.g. 20251115_12z.csv).

The NBM data can be found at: https://noaa-nbm-grib2-pds.s3.amazonaws.com/index.html

"""

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import warnings
!pip install boto3
!pip install botocore
import boto3
from botocore import UNSIGNED
from botocore.client import Config
import re
from datetime import datetime, timedelta

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 4.7 MB/s eta 0:00:00


# **Functions**

In [ ]:
def download_nbm_data(bucket_name, product, cycle_date, cycle_hour, subdirectory, local_dir):
    s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
    s3_prefix = f'{product}.{cycle_date}/{cycle_hour}/{subdirectory}/'

    # Define the exact folder for raw files
    raw_dir = os.path.join(local_dir, 'NBM_Raw_Data_Files')

    try:
        response = s3.list_objects_v2(Bucket=bucket_name, Prefix=s3_prefix)
        if 'Contents' not in response:
            return False

        for item in response['Contents']:
            s3_key = item['Key']
            # Using 'nbstx' as per your recent change
            if 'nbstx' in s3_key:
                file_name = f"{cycle_date}_{os.path.basename(s3_key)}"
                local_path = os.path.join(raw_dir, file_name)

                # Check if the FILE itself exists to avoid re-downloading
                if not os.path.exists(local_path):
                    print(f"Downloading {file_name}...")
                    s3.download_file(bucket_name, s3_key, local_path)
                else:
                    print(f"Skipping {file_name}, already exists.")
        return True
    except Exception as e:
        print(f"Download error: {e}")
        return False

def process_data(cycle_date, cycle_hour, local_dir, station_id):
    file_path = os.path.join(local_dir, 'NBM_Raw_Data_Files', f"{cycle_date}_blend_nbstx.t{cycle_hour}z")
    target_found = False
    nbm_run_time = None
    utc, tmp, dpt, sky, mix, wsp, wdr = [], [], [], [], [], [], []

    if not os.path.exists(file_path):
        # Ensure the number of returned values matches the unpacking in the main code
        return None, [], [], [], [], [], [], []

    try:
        with open(file_path, 'r') as file:
            for line in file:
                # Look for the specific station ID
                if station_id in line:
                    target_found = True
                    date_match = re.search(r'(\d{1,2}/\d{1,2}/\d{4})', line)
                    time_match = re.search(r'(\d{4}) UTC', line)
                    if date_match and time_match:
                        date_obj = datetime.strptime(date_match.group(1), '%m/%d/%Y')
                        nbm_run_time = f"{date_obj.strftime('%Y%m%d')}{time_match.group(1)[:2]}"

                if target_found:
                    line_fixed = line.replace('-', ' -') # The code struggles with negative numbers in the line with no spacing so we add a space
                    parts = line_fixed.strip().split()
                    if not parts: continue
                    tag = parts[0]
                    vals = parts[1:]

                    if 'UTC' in tag: utc = vals
                    elif 'DPT' in tag: dpt = vals
                    elif 'TMP' in tag: tmp = vals
                    elif 'SKY' in tag: sky = vals
                    elif 'WSP' in tag: wsp = vals
                    elif 'WDR' in tag: wdr = vals
                    elif 'MHT' in tag:
                        mix = vals
                        target_found = False # Reset flag
                        break # Exit the file loop immediately after finding the last variable

                    # End of station block check

    except Exception as e:
        print(f"Error reading {station_id} in {cycle_date}: {e}")

    return nbm_run_time, utc, tmp, dpt, sky, mix, wsp, wdr

# **Main Code**

In [ ]:
import os
from datetime import datetime, timedelta
import pandas as pd

################################## INPUTS ####################################
STATIONS = ['KCAR']
BUCKET_NAME = 'noaa-nbm-grib2-pds'
PRODUCT = 'blend'
SUBDIRECTORY = 'text'
CYCLE_HOUR = '13'
LOCAL_DIR = "/content/drive/MyDrive/Fire Weather Focal Point/TdAI_Project/Development/Data/NBM_Data"
RAW_DATA_DIR = os.path.join(LOCAL_DIR, "NBM_Raw_Data_Files")

YEARS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
START_MD = (2, 28)
END_MD = (11, 15)
##############################################################################

# Ensure directories exist
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# Create Date List
all_dates = []
for year in YEARS:
    curr = datetime(year, START_MD[0], START_MD[1])
    stop = datetime(year, END_MD[0], END_MD[1])
    while curr <= stop:
        all_dates.append(curr.strftime('%Y%m%d'))
        curr += timedelta(days=1)

# Initialize a dictionary of lists to hold rows for each station
station_data_store = {station: [] for station in STATIONS}

print(f"🚀 Processing {len(STATIONS)} stations over {len(all_dates)} days...")

for cycle_date in all_dates:
    # 1. Download file ONCE per date
    success = download_nbm_data(BUCKET_NAME, PRODUCT, cycle_date, CYCLE_HOUR, SUBDIRECTORY, LOCAL_DIR)
    if not success:
      print(f"❌ Failed {cycle_date}")
      continue

    # 2. Extract data for EACH station from that one file
    for sid in STATIONS:
        nbm_init_str, utc_l, tmp_1, dpt_l, sky_l, mix_l, wsp_l, wdr_l = process_data(cycle_date, CYCLE_HOUR, LOCAL_DIR, sid)

        # Validation (added check for wsp and wdr)
        if not nbm_init_str or not (len(utc_l) == len(tmp_1) == len(dpt_l) == len(sky_l) == len(mix_l) == len(wsp_l) == len(wdr_l)):
            continue

        try:
            nbm_init_time = datetime.strptime(nbm_init_str, '%Y%m%d%H')

            # --- NEW LOGIC: Define exact 21z targets for Day 1 and Day 2 ---
            # Day 1 is the same date as the initialization time
            day1_21z = datetime.combine(nbm_init_time.date(), datetime.min.time()) + timedelta(hours=21)
            # Day 2 is exactly 24 hours later
            day2_21z = day1_21z + timedelta(days=1)

            last_hr, days_added = -1, 0

            for fcst_hr, t, d, s, m, ws, wd in zip(utc_l, tmp_1, dpt_l, sky_l, mix_l, wsp_l, wdr_l):
                hr_int = int(fcst_hr)
                if last_hr != -1 and hr_int < last_hr: days_added += 1

                v_time = datetime.combine(nbm_init_time.date(), datetime.min.time()) + \
                         timedelta(days=days_added) + timedelta(hours=hr_int)

                if last_hr == -1 and hr_int < nbm_init_time.hour:
                    v_time += timedelta(days=1)
                    days_added += 1

                last_hr = hr_int

                # --- NEW LOGIC: Check if valid time matches Day 1 or Day 2 21z targets ---
                if v_time == day1_21z or v_time == day2_21z:
                    try:
                        station_data_store[sid].append({
                            'NBM Initialization Time (UTC)': nbm_init_time,
                            'NBM Forecast Valid (UTC)': v_time,
                            'NBM Temperature (F)': float(t),
                            'NBM Dewpoint (F)': float(d),
                            'NBM Cloud Cover (%)': float(s),
                            'NBM Mixing Height (100s ft AGL)': float(m),
                            'NBM Wind Speed (kts)': float(ws),
                            'NBM Wind Direction (deg)': float(wd)
                        })
                    except ValueError: continue
        except Exception as e:
            continue

# STEP 5: SAVE INDIVIDUAL FILES
for sid in STATIONS:
    if station_data_store[sid]:
        df = pd.DataFrame(station_data_store[sid])
        df = df.drop_duplicates(subset=['NBM Initialization Time (UTC)', 'NBM Forecast Valid (UTC)'])
        # Fixed typo: changed cycle_hour to CYCLE_HOUR
        output_path = os.path.join(LOCAL_DIR, f"NBM_Master_Data_{sid}_{CYCLE_HOUR}z.csv")
        df.to_csv(output_path, index=False)
        print(f"✅ {sid} saved: {len(df)} rows")
    else:
        print(f"❌ No data for {sid}")

🚀 Processing 1 stations over 1829 days...
❌ Failed 20200228
❌ Failed 20200229
❌ Failed 20200301
❌ Failed 20200302
❌ Failed 20200303
❌ Failed 20200304
❌ Failed 20200305
❌ Failed 20200306
❌ Failed 20200307
❌ Failed 20200308
❌ Failed 20200309
❌ Failed 20200310
❌ Failed 20200311
❌ Failed 20200312
❌ Failed 20200313
❌ Failed 20200314
❌ Failed 20200315
❌ Failed 20200316
❌ Failed 20200317
❌ Failed 20200318
❌ Failed 20200319
❌ Failed 20200320
❌ Failed 20200321
❌ Failed 20200322
❌ Failed 20200323
❌ Failed 20200324
❌ Failed 20200325
❌ Failed 20200326
❌ Failed 20200327
❌ Failed 20200328
❌ Failed 20200329
❌ Failed 20200330
❌ Failed 20200331
❌ Failed 20200401
❌ Failed 20200402
❌ Failed 20200403
❌ Failed 20200404
❌ Failed 20200405
❌ Failed 20200406
❌ Failed 20200407
❌ Failed 20200408
❌ Failed 20200409
❌ Failed 20200410
❌ Failed 20200411
❌ Failed 20200412
❌ Failed 20200413
❌ Failed 20200414
❌ Failed 20200415
❌ Failed 20200416
❌ Failed 20200417
❌ Failed 20200418
❌ Failed 20200419
❌ Failed 20200420
❌ Fa